# 第1章　事象と確率

統計学実践ワークブック 第1章に対応。
式を動かしながら・可視化しながらイメージを固める。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle
import matplotlib.patheffects as pe
from scipy import stats

plt.rcParams['font.family'] = 'IPAexGothic'
plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})

# 日本語フォントが無い環境用フォールバック
import matplotlib
try:
    matplotlib.font_manager.findfont('IPAexGothic', fallback_to_default=False)
except:
    plt.rcParams['font.family'] = 'DejaVu Sans'

---
## 1. 事象・余事象

**標本空間** $\Omega$：起こりうる全結果の集合

**事象** $A \subseteq \Omega$：結果の部分集合

**余事象** $A^c = \Omega \setminus A$：$A$ が起こらない事象

$$P(A^c) = 1 - P(A)$$

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.set_xlim(0, 6); ax.set_ylim(0, 4); ax.set_aspect('equal')
ax.axis('off')

# 標本空間 Ω
rect = mpatches.FancyBboxPatch((0.1, 0.1), 5.8, 3.8,
    boxstyle='round,pad=0.1', linewidth=2,
    edgecolor='black', facecolor='#d0e8f1', zorder=0)
ax.add_patch(rect)
ax.text(5.6, 3.7, r'$\Omega$', fontsize=16, ha='right', va='top', fontweight='bold')

# 事象 A
circle_A = Circle((2.5, 2), 1.2, color='#f4a261', alpha=0.8, zorder=1)
ax.add_patch(circle_A)
ax.text(2.5, 2, r'$A$', fontsize=18, ha='center', va='center', fontweight='bold')

# 余事象 Ac ラベル
ax.text(4.8, 3.0, r'$A^c$', fontsize=18, ha='center', va='center',
        color='#1d3557', fontweight='bold')

ax.set_title(r'事象 $A$ と余事象 $A^c = \Omega \setminus A$', fontsize=13)
plt.tight_layout()
plt.show()

# 数値で確認
P_A = 0.3
P_Ac = 1 - P_A
print(f'P(A)  = {P_A}')
print(f'P(Ac) = 1 - P(A) = {P_Ac}')

---
## 2. 包除原理

$$P(A \cup B) = P(A) + P(B) - P(A \cap B)$$

**直感**：$A + B$ を足すと重複部分 $A \cap B$ を2回数えてしまう。1回引いて補正する。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

def draw_two_circles(ax, alpha_A=0.5, alpha_B=0.5, title=''):
    ax.set_xlim(0, 6); ax.set_ylim(0, 4); ax.set_aspect('equal'); ax.axis('off')
    rect = mpatches.FancyBboxPatch((0.1, 0.1), 5.8, 3.8,
        boxstyle='round,pad=0.1', linewidth=1.5,
        edgecolor='black', facecolor='#f0f0f0')
    ax.add_patch(rect)
    cA = Circle((2.2, 2), 1.3, color='#e63946', alpha=alpha_A)
    cB = Circle((3.8, 2), 1.3, color='#457b9d', alpha=alpha_B)
    ax.add_patch(cA); ax.add_patch(cB)
    ax.text(1.5, 2, 'A', fontsize=16, ha='center', va='center', fontweight='bold', color='white')
    ax.text(4.5, 2, 'B', fontsize=16, ha='center', va='center', fontweight='bold', color='white')
    ax.set_title(title, fontsize=11)

draw_two_circles(axes[0], alpha_A=0.8, alpha_B=0, title='P(A) = 0.4')
draw_two_circles(axes[1], alpha_A=0, alpha_B=0.8, title='P(B) = 0.5')
draw_two_circles(axes[2], alpha_A=0.6, alpha_B=0.6, title='P(A∪B) = P(A)+P(B)-P(A∩B)')

plt.suptitle('包除原理：重複部分を引いて正確な確率に', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# 数値で確認
P_A, P_B, P_AB = 0.4, 0.5, 0.2
P_AuB = P_A + P_B - P_AB
print(f'P(A)       = {P_A}')
print(f'P(B)       = {P_B}')
print(f'P(A∩B)    = {P_AB}')
print(f'P(A∪B)    = {P_A} + {P_B} - {P_AB} = {P_AuB}')

---
## 3. 条件付き確率

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}, \quad P(B) > 0$$

**直感**：$B$ が起きたことがわかった瞬間、「世界」が $\Omega$ から $B$ に縮小する。
そこでの $A$ の割合が $P(A \mid B)$。

In [ ]:
# グリッドで面積として可視化
np.random.seed(42)
n = 10000

# P(A)=0.4, P(B)=0.5, P(A∩B)=0.2
# 4つの排反事象に分解: AB, A∩Bc, Ac∩B, AcBc
probs = [0.2, 0.2, 0.3, 0.3]  # P(AB), P(A∩Bc), P(Ac∩B), P(AcBc)
labels = ['A∩B', 'A∩Bc', 'Ac∩B', 'Ac∩Bc']
colors = ['#e63946', '#ffb4a2', '#457b9d', '#a8dadc']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# 左：全体の確率空間（面積図）
ax = axes[0]
# A の幅=0.4, B の高さ=0.5 で長方形分割
# x軸: [0,0.4] = A, [0.4,1.0] = Ac
# y軸: [0,0.5] = B, [0.5,1.0] = Bc
rects = [
    (0,   0,   0.4, 0.5, '#e63946', 'P(A∩B)=0.20'),   # A∩B
    (0,   0.5, 0.4, 0.5, '#ffb4a2', 'P(A∩Bc)=0.20'),  # A∩Bc
    (0.4, 0,   0.6, 0.5, '#457b9d', 'P(Ac∩B)=0.30'),  # Ac∩B
    (0.4, 0.5, 0.6, 0.5, '#a8dadc', 'P(Ac∩Bc)=0.30'), # Ac∩Bc
]
for x, y, w, h, c, lbl in rects:
    ax.add_patch(mpatches.Rectangle((x, y), w, h, color=c, alpha=0.8, ec='white', lw=2))
    ax.text(x+w/2, y+h/2, lbl, ha='center', va='center', fontsize=9, fontweight='bold')

ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_xlabel('A ← 0.4 →  ← 0.6 → Ac', fontsize=10)
ax.set_ylabel('B ← 0.5 →  ← 0.5 → Bc', fontsize=10)
ax.set_title('確率空間の面積分割', fontsize=12)

# 右：B に絞ったとき（条件付き確率）
ax2 = axes[1]
# B の中だけ → 高さ0.5の帯を1.0に正規化
# P(A|B) = P(A∩B)/P(B) = 0.2/0.5 = 0.4
P_B = 0.5
rects_B = [
    (0,   0, 0.4, 1.0, '#e63946', f'P(A|B)={0.2/P_B:.2f}'),
    (0.4, 0, 0.6, 1.0, '#457b9d', f'P(Ac|B)={0.3/P_B:.2f}'),
]
for x, y, w, h, c, lbl in rects_B:
    ax2.add_patch(mpatches.Rectangle((x, y), w, h, color=c, alpha=0.8, ec='white', lw=2))
    ax2.text(x+w/2, y+h/2, lbl, ha='center', va='center', fontsize=11, fontweight='bold')

ax2.set_xlim(0,1); ax2.set_ylim(0,1)
ax2.set_xlabel('A ← 0.4 →  ← 0.6 → Ac', fontsize=10)
ax2.set_title('B が起きた条件下（世界が B に縮小）', fontsize=12)

plt.suptitle('条件付き確率：P(A|B) = P(A∩B) / P(B)', fontsize=13)
plt.tight_layout()
plt.show()

P_A_given_B = 0.2 / 0.5
print(f'P(A|B) = P(A∩B) / P(B) = 0.2 / 0.5 = {P_A_given_B}')

---
## 4. ベイズの定理・事前確率・事後確率

$$P(H \mid D) = \frac{P(D \mid H)\, P(H)}{P(D)}$$

| 項 | 名前 | 意味 |
|---|---|---|
| $P(H)$ | **事前確率** | データを見る前の信念 |
| $P(D \mid H)$ | **尤度** | 仮説 $H$ のもとでデータ $D$ が出る確率 |
| $P(H \mid D)$ | **事後確率** | データを見た後に更新された信念 |
| $P(D)$ | **周辺尤度** | 正規化定数（$=\sum_H P(D|H)P(H)$） |

**具体例**：疾患検査
- 疾患の有病率（事前確率）$P(H) = 0.01$
- 検査の感度（陽性の場合に正しく検出）$P(D \mid H) = 0.95$
- 偽陽性率 $P(D \mid H^c) = 0.05$

In [ ]:
def bayes_update(prior, likelihood, false_positive):
    P_D = likelihood * prior + false_positive * (1 - prior)  # 全確率
    posterior = (likelihood * prior) / P_D
    return posterior

# 事前確率を変えたとき事後確率がどう変わるか
priors = np.linspace(0.001, 0.5, 200)
sensitivity = 0.95   # P(D|H)
fpr = 0.05           # P(D|Hc)

posteriors = [bayes_update(p, sensitivity, fpr) for p in priors]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左：事前 vs 事後
ax = axes[0]
ax.plot(priors, posteriors, color='#e63946', lw=2.5, label='事後確率 P(H|D)')
ax.plot(priors, priors, 'k--', lw=1, alpha=0.5, label='事前=事後（対角線）')

# 具体点：有病率1%
p0 = 0.01
post0 = bayes_update(p0, sensitivity, fpr)
ax.scatter([p0], [post0], color='#e63946', s=80, zorder=5)
ax.annotate(f'有病率1%\n→ 事後確率 {post0:.1%}',
            xy=(p0, post0), xytext=(0.08, 0.3),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=10, color='#e63946')

ax.set_xlabel('事前確率 P(H)', fontsize=11)
ax.set_ylabel('事後確率 P(H|D)', fontsize=11)
ax.set_title(f'ベイズ更新（感度={sensitivity}, 偽陽性率={fpr}）', fontsize=11)
ax.legend(fontsize=9)

# 右：棒グラフで「検査陽性 1000人の内訳」を可視化
ax2 = axes[1]
N = 10000
prevalence = 0.01
true_pos  = N * prevalence * sensitivity          # 真陽性
false_pos = N * (1 - prevalence) * fpr            # 偽陽性
total_pos = true_pos + false_pos

bars = ax2.bar(['真陽性\n(病気あり・陽性)', '偽陽性\n(病気なし・陽性)'],
               [true_pos, false_pos],
               color=['#e63946', '#457b9d'], alpha=0.8, width=0.5)
for bar, val in zip(bars, [true_pos, false_pos]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f'{val:.0f}人', ha='center', fontsize=11)

ax2.set_ylabel('人数（N=10,000人中）', fontsize=11)
ax2.set_title(f'陽性判定 {total_pos:.0f}人の内訳\n→ 本当に病気なのは {true_pos/total_pos:.1%}', fontsize=11)
ax2.set_ylim(0, false_pos * 1.3)

plt.suptitle('ベイズの定理：有病率1%でも検査陽性のうち本当の病気はわずか', fontsize=12)
plt.tight_layout()
plt.show()

print(f'P(H|D) = {post0:.4f}  ← 有病率1%、感度95%、偽陽性率5% の場合')

---
## 5. 条件付き独立

**独立**：$A \perp B \iff P(A \cap B) = P(A)P(B)$

**条件付き独立**：$A \perp B \mid C$
$$P(A \cap B \mid C) = P(A \mid C)\,P(B \mid C)$$

**直感**：$C$ が既知になると $A$ と $B$ のつながりが切れる（$C$ が「原因」のとき）。

例：$C$=雨、$A$=地面が濡れる、$B$=傘を持っている人が多い  
→ $C$ を知らないと $A,B$ は相関する。$C$（雨かどうか）がわかると独立。

In [ ]:
np.random.seed(0)
n = 2000

# C: 雨(1) or 晴れ(0)
C = np.random.binomial(1, 0.4, n)

# A: 地面が濡れる (雨の日は確率高い)
A = np.where(C == 1,
             np.random.binomial(1, 0.9, n),
             np.random.binomial(1, 0.1, n))

# B: 傘を持つ人が多い (雨の日は確率高い)
B = np.where(C == 1,
             np.random.binomial(1, 0.85, n),
             np.random.binomial(1, 0.15, n))

# 全体での相関
corr_all = np.corrcoef(A, B)[0, 1]

# C=1 (雨) の中での相関
mask_rain = C == 1
corr_rain = np.corrcoef(A[mask_rain], B[mask_rain])[0, 1]

# C=0 (晴れ) の中での相関
mask_sun = C == 0
corr_sun = np.corrcoef(A[mask_sun], B[mask_sun])[0, 1]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
titles = ['全体（C未知）', 'C=1（雨）', 'C=0（晴れ）']
masks = [np.ones(n, bool), mask_rain, mask_sun]
corrs = [corr_all, corr_rain, corr_sun]
colors = ['#6a4c93', '#e63946', '#f4a261']

for ax, title, mask, corr, col in zip(axes, titles, masks, corrs, colors):
    jitter = lambda x: x + np.random.uniform(-0.1, 0.1, len(x))  # len(x) で配列長を取得
    ax.scatter(jitter(A[mask]), jitter(B[mask]),
               alpha=0.15, s=15, color=col)
    ax.set_xlabel('A (地面が濡れる)', fontsize=9)
    ax.set_ylabel('B (傘が多い)', fontsize=9)
    ax.set_title(f'{title}\n相関係数 r={corr:.3f}', fontsize=10)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])

plt.suptitle('条件付き独立：C（雨）が既知になると A と B の相関が消える', fontsize=12)
plt.tight_layout()
plt.show()

print(f'全体相関     : {corr_all:.3f}  ← A と B は相関している')
print(f'C=1(雨)相関  : {corr_rain:.3f}  ← C がわかると無相関に近い')
print(f'C=0(晴れ)相関: {corr_sun:.3f}  ← C がわかると無相関に近い')

---
## 6. 確率変数

**確率変数** $X$：標本空間 $\Omega$ の各結果を実数に対応させる関数
$$X : \Omega \to \mathbb{R}$$

- **離散型**：$X$ が有限/可算無限の値をとる（例：さいころ）
- **連続型**：$X$ が実数の区間の値をとる（例：身長）

エンジニア的に言えば：**ランダムな出力を持つ関数**。関数の戻り値が毎回ランダムに変わるイメージ。

In [ ]:
np.random.seed(42)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 離散型：さいころ
ax1 = axes[0]
die_values = np.arange(1, 7)
die_probs  = np.ones(6) / 6
ax1.bar(die_values, die_probs, color='#457b9d', alpha=0.8, width=0.6,
        edgecolor='white', linewidth=1.5)
ax1.set_xlabel('X の値', fontsize=11)
ax1.set_ylabel('P(X = x)', fontsize=11)
ax1.set_title('離散型確率変数\nX = さいころの目（各 1/6）', fontsize=11)
ax1.set_ylim(0, 0.3)
ax1.set_xticks(die_values)

# 連続型：標準正規分布
ax2 = axes[1]
x = np.linspace(-4, 4, 500)
pdf = stats.norm.pdf(x)
ax2.plot(x, pdf, color='#e63946', lw=2.5)
ax2.fill_between(x, pdf, alpha=0.2, color='#e63946')

# P(-1 < X < 1) を塗る
x_fill = np.linspace(-1, 1, 200)
ax2.fill_between(x_fill, stats.norm.pdf(x_fill),
                 alpha=0.5, color='#e63946', label=f'P(-1<X<1) = {stats.norm.cdf(1)-stats.norm.cdf(-1):.3f}')
ax2.set_xlabel('X の値', fontsize=11)
ax2.set_ylabel('確率密度 f(x)', fontsize=11)
ax2.set_title('連続型確率変数\nX ～ N(0, 1)（標準正規分布）', fontsize=11)
ax2.legend(fontsize=10)

plt.suptitle('確率変数：離散型 vs 連続型', fontsize=13)
plt.tight_layout()
plt.show()

---
## 7. 期待値・分散

**期待値**（重み付き平均）：
$$E[X] = \sum_x x\, P(X=x) \quad \text{（離散）}, \qquad E[X] = \int x\, f(x)\, dx \quad \text{（連続）}$$

**分散**（ばらつきの大きさ）：
$$\text{Var}(X) = E[(X - E[X])^2] = E[X^2] - (E[X])^2$$

**直感**：期待値は「重心」、分散は「重心からの平均二乗距離」。

In [ ]:
# さいころで手計算 vs numpy
x_vals = np.arange(1, 7)
probs   = np.ones(6) / 6

E_X   = np.sum(x_vals * probs)
E_X2  = np.sum(x_vals**2 * probs)
Var_X = E_X2 - E_X**2
Std_X = np.sqrt(Var_X)

print(f'E[X]       = {E_X:.4f}  (理論値 3.5)')
print(f'E[X²]      = {E_X2:.4f}')
print(f'Var(X)     = E[X²] - E[X]² = {E_X2:.4f} - {E_X**2:.4f} = {Var_X:.4f}  (理論値 35/12 ≈ 2.917)')
print(f'Std(X)     = {Std_X:.4f}')

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax1 = axes[0]
bars = ax1.bar(x_vals, probs, color='#457b9d', alpha=0.7, width=0.6, edgecolor='white')
ax1.axvline(E_X, color='#e63946', lw=2.5, linestyle='--', label=f'E[X]={E_X:.1f}（重心）')
ax1.set_xlabel('X', fontsize=11); ax1.set_ylabel('P(X=x)', fontsize=11)
ax1.set_title('さいころ：期待値は「重心」', fontsize=11)
ax1.legend(fontsize=10)

# 分散の直感：(x - E[X])² の重み付き平均
ax2 = axes[1]
deviations = (x_vals - E_X)**2
ax2.bar(x_vals, deviations * probs, color='#f4a261', alpha=0.8, width=0.6, edgecolor='white',
        label=r'$(x - E[X])^2 \cdot P(X=x)$')
ax2.axhline(Var_X, color='#e63946', lw=2, linestyle='--', label=f'Var(X)={Var_X:.3f}（これらの合計）')
ax2.set_xlabel('X', fontsize=11); ax2.set_ylabel(r'$(x - E[X])^2 \cdot P(X=x)$', fontsize=10)
ax2.set_title('分散＝「重心からの距離の二乗」の期待値', fontsize=11)
ax2.legend(fontsize=9)

plt.suptitle('期待値と分散の直感', fontsize=13)
plt.tight_layout()
plt.show()

---
## 8. 確率密度関数 (PDF)

連続型確率変数 $X$ の**確率密度関数** $f(x)$ は：

$$P(a \le X \le b) = \int_a^b f(x)\, dx$$

条件：$f(x) \ge 0$、$\int_{-\infty}^{\infty} f(x)\, dx = 1$

**重要**：$f(x)$ は確率**ではない**。$x$ 付近の確率の**密度**（濃さ）。

- $P(X = x) = 0$（1点の確率はゼロ）
- 確率は**面積**として計算する

**累積分布関数 (CDF)**：$F(x) = P(X \le x) = \int_{-\infty}^{x} f(t)\, dt$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

x = np.linspace(-4, 4, 500)
mu, sigma = 0, 1
pdf = stats.norm.pdf(x, mu, sigma)
cdf = stats.norm.cdf(x, mu, sigma)

# 左：PDF + 面積として確率を可視化
ax1 = axes[0]
ax1.plot(x, pdf, color='#457b9d', lw=2.5, label='f(x) = PDF')

# P(-1 < X < 1)
a, b = -1, 1
x_fill = np.linspace(a, b, 200)
prob_ab = stats.norm.cdf(b, mu, sigma) - stats.norm.cdf(a, mu, sigma)
ax1.fill_between(x_fill, stats.norm.pdf(x_fill, mu, sigma),
                 alpha=0.45, color='#e63946',
                 label=f'P(-1<X<1) = 面積 = {prob_ab:.3f}')
ax1.axvline(a, color='#e63946', lw=1.5, linestyle=':')
ax1.axvline(b, color='#e63946', lw=1.5, linestyle=':')
ax1.set_xlabel('x', fontsize=11); ax1.set_ylabel('f(x)', fontsize=11)
ax1.set_title('PDF：確率 = グラフの面積', fontsize=12)
ax1.legend(fontsize=10)

# 右：CDF
ax2 = axes[1]
ax2.plot(x, cdf, color='#e63946', lw=2.5, label='F(x) = CDF')
# P(X <= 1) の点を強調
for xv, col, lbl in [(-1, '#6a4c93', 'F(-1)'), (0, '#f4a261', 'F(0)'), (1, '#2a9d8f', 'F(1)')]:
    yv = stats.norm.cdf(xv)
    ax2.scatter([xv], [yv], s=70, zorder=5, color=col)
    ax2.annotate(f'{lbl}={yv:.3f}', xy=(xv, yv),
                 xytext=(xv + 0.5, yv - 0.08),
                 fontsize=9, color=col,
                 arrowprops=dict(arrowstyle='->', color=col))

ax2.set_xlabel('x', fontsize=11); ax2.set_ylabel('F(x) = P(X ≤ x)', fontsize=11)
ax2.set_title('CDF：累積分布関数（単調増加、0→1）', fontsize=12)
ax2.legend(fontsize=10)

plt.suptitle('確率密度関数 (PDF) と累積分布関数 (CDF)', fontsize=13)
plt.tight_layout()
plt.show()

# 数値で確認
print(f'P(-1 < X < 1) = F(1) - F(-1) = {stats.norm.cdf(1):.4f} - {stats.norm.cdf(-1):.4f} = {prob_ab:.4f}')
print(f'全体の積分    = {np.trapz(pdf, x):.6f}  ≈ 1.0 （確認）')

---
## まとめ

| 概念 | 式 | イメージ |
|---|---|---|
| 余事象 | $P(A^c)=1-P(A)$ | Ω から A を引いた残り |
| 包除原理 | $P(A\cup B)=P(A)+P(B)-P(A\cap B)$ | 重複を1回引く |
| 条件付き確率 | $P(A\mid B)=P(A\cap B)/P(B)$ | 世界を B に縮小 |
| ベイズの定理 | $P(H\mid D)\propto P(D\mid H)P(H)$ | データで信念を更新 |
| 条件付き独立 | $P(A\cap B\mid C)=P(A\mid C)P(B\mid C)$ | C が既知なら A,B は無関係 |
| 期待値 | $E[X]=\sum x P(x)$ | 重心 |
| 分散 | $\text{Var}(X)=E[(X-E[X])^2]$ | 重心からの平均二乗距離 |
| PDF | $P(a\le X\le b)=\int_a^b f(x)dx$ | 確率 = 面積 |

**次章**：第2章「確率分布と母関数」